In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy.sparse import coo_matrix
from PIL import Image
import matplotlib.pyplot as plt
from scipy.linalg import lu, lu_factor, lu_solve, inv

# Теория

## Векторы

In [11]:
# Для создания вектора с заданными значениями
# используется функция `np.array()`
v = np.array([2, 5])  # array([2, 5])

# Также можно создавать специальные векторы.
# Например, вектор нулей заданного размера,
# используя функцию `np.zeros()`
zeros = np.zeros(3)  # array([0., 0., 0.])

# Или вектор с равномерным шагом (аналог функции range),
# используя функцию `np.arange()`
a = np.arange(0, 10, 2)  # array([0, 2, 4, 6, 8])


In [12]:

def cosine_similarity(vec1: np.array, vec2: np.array) -> float:
    """Вычисляет косинусную меру двух векторов."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def evaulate_result(embeddings: np.array, result_embedding: np.array) -> None:
    """Вычисляет косинусное сходство результата с каждым словом в словаре."""
    print("Косинусное сходство результата с каждым словом:")
    for word, vec in embeddings.items():
		    print(f"{word}: {cosine_similarity(result_embedding, vec):.3f}")

# Пример 1: ebook ≈ book - pages + digital
embeddings = {
    "book":     np.array([0.5, 0.7, 0.6, 0.4, 0.3]),
    "pages":    np.array([0.2, 0.3, 0.1, 0.0, 0.2]),
    "digital":  np.array([0.1, 0.2, 0.4, 0.3, 0.4]),
    "ebook":    np.array([0.3, 0.6, 1.0, 0.7, 0.5]),
    "magazine": np.array([0.3, 0.5, 0.5, 0.2, 0.1])
}

result = embeddings["book"] - embeddings["pages"] + embeddings["digital"]
evaulate_result(embeddings, result)

# Output:
# Косинусное сходство результата с каждым словом:
# book: 0.951
# pages: 0.737
# digital: 0.953
# ebook: 0.996
# magazine: 0.921


Косинусное сходство результата с каждым словом:
book: 0.951
pages: 0.737
digital: 0.953
ebook: 0.996
magazine: 0.921


In [13]:
# Пример 2: hybrid ≈ motor - gasoline + electricity

embeddings = {
    "motor":       np.array([0.6, 0.7, 0.4, 0.2, 0.3]),
    "gasoline":    np.array([0.2, 0.3, 0.1, 0.0, 0.1]),
    "electricity": np.array([0.4, 0.5, 0.3, 0.4, 0.3]),
    "hybrid":      np.array([0.8, 0.9, 0.7, 0.6, 0.5]),
    "battery":     np.array([0.1, 0.4, 0.2, 0.5, 0.2])
}

result = embeddings["motor"] - embeddings["gasoline"] + embeddings["electricity"]
evaulate_result(embeddings, result)

# Output:
# Косинусное сходство результата с каждым словом:
# motor: 0.975
# gasoline: 0.896
# electricity: 0.995
# hybrid: 0.998
# battery: 0.873



Косинусное сходство результата с каждым словом:
motor: 0.975
gasoline: 0.896
electricity: 0.995
hybrid: 0.998
battery: 0.873


In [14]:
# Пример 3:  profit ≈ investment - risk + opportunity

embeddings = {
    "investment":   np.array([0.7, 0.8, 0.6, 0.7, 0.5]),
    "risk":         np.array([0.3, 0.4, 0.2, 0.2, 0.3]),
    "opportunity":  np.array([0.5, 0.6, 0.4, 0.5, 0.6]),
    "profit":       np.array([0.9, 0.9 , 0.8, 1. , 0.8]),
    "loss":         np.array([-0.6, -0.7, -0.5, -0.6, -0.4])
}

result = embeddings["investment"] - embeddings["risk"] + embeddings["opportunity"]
evaulate_result(embeddings, result)

# Output:
# Косинусное сходство результата с каждым словом:
# investment: 0.997
# risk: 0.969
# opportunity: 0.989
# profit: 0.999
# loss: -0.995



Косинусное сходство результата с каждым словом:
investment: 0.997
risk: 0.969
opportunity: 0.989
profit: 0.999
loss: -0.995


## Матрицы

In [15]:
# Для создания матрицы с заданными значениями
# используется функция `np.array()`, как и для векторов,
# но, в отличие от векторов, матрицы могут быть многомерными

A = np.array([[1, 2], [3, 4], [5, 6]])  # матрица размера 3x2
print(A)

# Output:
# [[1 2]
# [3 4]
# [5 6]]

O = np.zeros(shape=(3, 2))
print(O)

# Output:
# [[0. 0.]
# [0. 0.]
# [0. 0.]]

E = np.ones(shape=(3, 2))
print(E)

# Output:
# [[1. 1.]
# [1. 1.]
# [1. 1.]]


rng = np.random.default_rng()
R = rng.random(shape=(3, 2))
print(R)

# Output:
# [[0.47821453 0.28421059]
# [0.2178972  0.07891322]
# [0.63259903 0.4502506 ]]

print(A[0, 1])
print(A[1:3])
print(A[0:2, 0])

# Output:
# 2
# array([[3, 4],
#       [5, 6]])
# array([1, 3])



[[1 2]
 [3 4]
 [5 6]]
[[0. 0.]
 [0. 0.]
 [0. 0.]]
[[1. 1.]
 [1. 1.]
 [1. 1.]]


TypeError: random() got an unexpected keyword argument 'shape'

In [16]:
# Создаём матрицу в виде списка списков
A = [
    [1, 2],
    [3, 4],
    [5, 6]
]

# Преобразуем список списков в DataFrame и задаём имена столбцов
df = pd.DataFrame(A, columns=["Column1", "Column2"])

# Выводим DataFrame (матрицу)
print(df)

# Output:
#     Column1  Column2
#  0        1        2
#  1        3        4
#  2        5        6


   Column1  Column2
0        1        2
1        3        4
2        5        6


In [17]:
# Группировка и агрегация данных по категориальному признаку
# Допустим, у нас есть данные о зарплатах по отделам
data_group = {
    'Отдел': ['HR', 'IT', 'HR', 'IT', 'HR'],
    'Зарплата': [40000, 70000, 45000, 72000, 42000]
}
df_group = pd.DataFrame(data_group)
# Группируем по столбцу 'Отдел' и 
# вычисляем среднюю зарплату в каждом отделе
grouped = df_group.groupby('Отдел').mean()
print("Средняя зарплата по отделам:")
print(grouped)

# Output:
# Средняя зарплата по отделам:
#            Зарплата
# Отдел              
# HR     42333.333333
# IT     71000.000000


Средняя зарплата по отделам:
           Зарплата
Отдел              
HR     42333.333333
IT     71000.000000


In [18]:
# Определяем ненулевые элементы: их значения и позиции (строки и столбцы)
rows = np.array([0, 1, 2, 0])
cols = np.array([1, 2, 0, 2])
data = np.array([10, 20, 30, 40])

# Создаём матрицу размером 3x3
sparse_coo = coo_matrix((data, (rows, cols)), shape=(3, 3))

print("Исходная матрица:")
print(sparse_coo.toarray())

print("\nРазрежённая матрица в формате COO:")
print(sparse_coo)

# Output:
# Исходная матрица:
# [[ 0 10 40]
#  [ 0  0 20]
#  [30  0  0]]
 
# Разрежённая матрица в формате COO:
#   (0, 1)	10
#   (1, 2)	20
#   (2, 0)	30
#   (0, 2)	40


Исходная матрица:
[[ 0 10 40]
 [ 0  0 20]
 [30  0  0]]

Разрежённая матрица в формате COO:
<COOrdinate sparse matrix of dtype 'int64'
	with 4 stored elements and shape (3, 3)>
  Coords	Values
  (0, 1)	10
  (1, 2)	20
  (2, 0)	30
  (0, 2)	40


In [19]:
# Шаг 1: Создаём базовый NumPy array
data = np.array([[0, 0, 3],
                 [4, 0, 0],
                 [0, 5, 6]])
print("Исходный NumPy array:")
print(data)

# Шаг 2: Преобразуем NumPy array в разрежённую матрицу в формате COO
sparse_matrix = coo_matrix(data)
print("\nРазрежённая матрица (COO формат):")
print(sparse_matrix)

# Шаг 3: Преобразуем разрежённую матрицу обратно в плотный NumPy array
dense_array = sparse_matrix.toarray()
print("\nПлотный NumPy array, полученный из разрежённой матрицы:")
print(dense_array)

# Шаг 4: Создаём DataFrame Pandas на основе плотного массива
df = pd.DataFrame(dense_array, columns=['Column1', 'Column2', 'Column3'])
print("\nDataFrame (Pandas):")
print(df)

# Output:
# Исходный NumPy array:
# [[0 0 3]
#  [4 0 0]
#  [0 5 6]]

# Разрежённая матрица (COO формат):
#   (0, 2)	3
#   (1, 0)	4
#   (2, 1)	5
#   (2, 2)	6

# Плотный NumPy array, полученный из разрежённой матрицы:
# [[0 0 3]
#  [4 0 0]
#  [0 5 6]]

# DataFrame (Pandas):
#    Column1  Column2  Column3
# 0        0        0        3
# 1        4        0        0
# 2        0        5        6


Исходный NumPy array:
[[0 0 3]
 [4 0 0]
 [0 5 6]]

Разрежённая матрица (COO формат):
<COOrdinate sparse matrix of dtype 'int64'
	with 4 stored elements and shape (3, 3)>
  Coords	Values
  (0, 2)	3
  (1, 0)	4
  (2, 1)	5
  (2, 2)	6

Плотный NumPy array, полученный из разрежённой матрицы:
[[0 0 3]
 [4 0 0]
 [0 5 6]]

DataFrame (Pandas):
   Column1  Column2  Column3
0        0        0        3
1        4        0        0
2        0        5        6


In [20]:
r = 4  # скаляр
v = np.arange(3)  # вектор 
a = np.arange(9).reshape(3, 3)  # матрица

# Сумма скаляра и вектора
print(r + v)

# Сумма вектора и матрицы
print(v + a)

# Сумма векторов разного размера
print(v.reshape(3, 1) + v) 

# Output:
# array([4, 5, 6])

# array([[ 0,  2,  4],
#        [ 3,  5,  7],
#        [ 6,  8, 10]])
       
# array([[0, 1, 2],
#        [1, 2, 3],
#        [2, 3, 4]])


[4 5 6]
[[ 0  2  4]
 [ 3  5  7]
 [ 6  8 10]]
[[0 1 2]
 [1 2 3]
 [2 3 4]]


In [21]:
# Задаём два вектора
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Поэлементное умножение: возвращает массив тех же размеров
elementwise = a * b 
print("Поэлементное умножение a * b:")
print(elementwise)

# Задаём две матрицы
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Поэлементное умножение: перемножаются соответствующие элементы
elementwise = A * B  
print("Поэлементное умножение A * B:")
print(elementwise)

# Output:
# Поэлементное умножение a * b:
# [ 4 10 18]
# Поэлементное умножение A * B:
# [[ 5 12]
#  [21 32]]


Поэлементное умножение a * b:
[ 4 10 18]
Поэлементное умножение A * B:
[[ 5 12]
 [21 32]]


In [22]:
# Задаём два вектора
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Матричное умножение: возвращает скаляр
dot_product = a @ b  # Результат: 1*4 + 2*5 + 3*6 = 32
print("Скалярное произведение a @ b:")
print(dot_product)

# Задаём две матрицы
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Матричное умножение (стандартное): используется оператор @
matrix_product = A @ B  
print("Матричное умножение A @ B:")
print(matrix_product)

# Output:
# Скалярное произведение a @ b:
# 32
# Матричное умножение A @ B:
# [[19 22]
#  [43 50]]


Скалярное произведение a @ b:
32
Матричное умножение A @ B:
[[19 22]
 [43 50]]


In [23]:
# Создаём два 3D массива (тензора)
X = np.random.rand(2, 3, 4)
Y = np.random.rand(2, 4, 5)

# np.matmul (или оператор @) умножает X и Y по последним двум измерениям
matmul_result = X @ Y  
print("Форма результата матричного умножения:", matmul_result.shape)

# Поэлементное умножение (*) здесь не сработает напрямую, 
# т. к. размеры (2,3,4) и (2,4,5) несовместимы для 
# поэлементного умножения без явного преобразования.
elementwise_result = X * Y

# Output:
# Форма результата матричного умножения: (2, 3, 5)
# ValueError: operands could not be broadcast together with shapes (2,3,4) (2,4,5)


Форма результата матричного умножения: (2, 3, 5)


ValueError: operands could not be broadcast together with shapes (2,3,4) (2,4,5) 

In [24]:
# Путь к изображению
image_path = "image.jpg"

# Считываем изображение и переводим его в оттенки серого
image = Image.open(image_path).convert("L")
image_np = np.array(image, dtype=float)

# Фильтр понижения яркости: отнимаем константу от всех элементов матрицы
brightness_offset = -50
result_brightness = np.clip(image_np + brightness_offset, 0, 255)

# Фильтр инверсии: вычисляем отрицательное изображение
result_inversion = 255 - image_np

# Пороговая фильтрация: если значение пикселя больше порога, ставим 255, иначе 0
threshold = 128
result_threshold = np.where(image_np > threshold, 255, 0)

# Создаём фигуру с заданными параметрами
fig, axes = plt.subplots(2, 2, figsize=(15, 8), dpi=200)

axes[0, 0].imshow(image_np, cmap='gray')
axes[0, 0].set_title("Исходное изображение")
axes[0, 0].axis('off')

axes[0, 1].imshow(result_brightness, cmap='gray')
axes[0, 1].set_title("Понижение яркости")
axes[0, 1].axis('off')

axes[1, 0].imshow(result_inversion, cmap='gray')
axes[1, 0].set_title("Инверсия изображения")
axes[1, 0].axis('off')

axes[1, 1].imshow(result_threshold, cmap='gray')
axes[1, 1].set_title("Пороговая фильтрация")
axes[1, 1].axis('off')

plt.subplots_adjust(wspace=-0.35, hspace=0.1)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'image.jpg'

## Линейные уравнения. Основы и Продвинуые методы

In [25]:
# Матрица коэффициентов
A = np.array([
    [1, 1, 1],
    [2, -1, 3],
    [-1, 4, 2]
], dtype=float)

# Вектор свободных членов
b = np.array([6, 8, 10], dtype=float)

# Решаем систему Ax = b
x = np.linalg.solve(A, b)

print("Решение системы (x, y):", x)

# Output:
# Решение системы: [2. 2. 2.]


Решение системы (x, y): [2. 2. 2.]


In [26]:
# Матрица коэффициентов
A = np.array([
    [1, 1, 1],
    [2, -1, 3],
    [-1, 4, 2]
], dtype=float)

# Вектор свободных членов
b = np.array([6, 8, 10], dtype=float)

# Решаем систему уравнений Ax = b
x = scipy.linalg.solve(A, b)

print("Решение системы:", x)

# Output:
# Решение системы: [2. 2. 2.]


Решение системы: [2. 2. 2.]


In [27]:
# Определяем матрицу A и правую часть b
A = np.array([[0, 1, 0],
               [1, 1, 2],
               [0, 3, 1]], dtype=float)
b = np.array([0, 14, 17], dtype=float)
    
# 1. Вычисляем PLU-разложение с помощью lu_factor
lu_and_piv = lu_factor(A)
print(f"Комбинированная матрица LU: \n {lu_and_piv[0]}")
print(f"Индексы перестановок: \n {lu_and_piv[1]}")
    
# 2. Решаем систему Ax = b, используя полученную факторизацию
x = lu_solve(lu_and_piv, b)
print(f"Решение системы Ax = b: \n {x}")
    
# 3. Вычисляем обратную матрицу A^{-1} с помощью scipy.linalg.inv
A_inv = inv(A)
print(f"Найденная обратная матрица A^{-1}: \n {A_inv}")
    
# Проверяем, что A * A_inv = E
res = np.allclose(np.eye(A.shape[0]), A @ A_inv)
print(f"Проверка: 'A * A_inv = E' is {res}")
    
# 4. Матрицы P, L, U можно получить с помощью scipy.linalg.lu
P, L, U = lu(A)
print(f"Перестановочная матрица P: \n {P}")
print(f"Нижнетреугольная матрица L: \n {L}")
print(f"Верхнетреугольная матрица U: \n {U}")
    
# Проверяем, что A = PLU
res = np.allclose(A, P @ L @ U)
print(f"Проверка: 'A = PLU' is {res}")
    
    # Output:
    # Комбинированная матрица LU: 
    #  [[ 1.          1.          2.        ]
    #  [ 0.          3.          1.        ]
    #  [ 0.          0.33333333 -0.33333333]]
    # Индексы перестановок: 
    #  [1 2 2]
    # Решение системы Ax = b: 
    #  [-20.   0.  17.]
    # Найденная обратная матрица A^-1: 
    #  [[ 5.  1. -2.]
    #  [ 1.  0.  0.]
    #  [-3.  0.  1.]]
    # Проверка: 'A * A_inv = E' is True
    # Перестановочная матрица P: 
    #  [[0. 0. 1.]
    #  [1. 0. 0.]
    #  [0. 1. 0.]]
    # Нижнетреугольная матрица L: 
    #  [[1.         0.         0.        ]
    #  [0.         1.         0.        ]
    #  [0.         0.33333333 1.        ]]
    # Верхнетреугольная матрица U: 
    #  [[ 1.          1.          2.        ]
    #  [ 0.          3.          1.        ]
    #  [ 0.          0.         -0.33333333]]
    # Проверка: 'A = PLU' is True

Комбинированная матрица LU: 
 [[ 1.          1.          2.        ]
 [ 0.          3.          1.        ]
 [ 0.          0.33333333 -0.33333333]]
Индексы перестановок: 
 [1 2 2]
Решение системы Ax = b: 
 [-20.   0.  17.]
Найденная обратная матрица A^-1: 
 [[ 5.  1. -2.]
 [ 1.  0.  0.]
 [-3.  0.  1.]]
Проверка: 'A * A_inv = E' is True
Перестановочная матрица P: 
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
Нижнетреугольная матрица L: 
 [[1.         0.         0.        ]
 [0.         1.         0.        ]
 [0.         0.33333333 1.        ]]
Верхнетреугольная матрица U: 
 [[ 1.          1.          2.        ]
 [ 0.          3.          1.        ]
 [ 0.          0.         -0.33333333]]
Проверка: 'A = PLU' is True


In [28]:
    # 1. Задаём A и b
    A = np.array([[4,12,-16],[12,37,-43],[-16,-43,98]], dtype=float)
    b = np.array([1,2,3], dtype=float)
    
    # 2. Разложение Холецкого
    L = np.linalg.cholesky(A)  # A = L @ L.T
    
    # 3. Решение A x = b
    y = np.linalg.solve(L, b)       # прямой ход
    x = np.linalg.solve(L.T, y)     # обратный ход
    
    # 4. Обратная матрица и определитель
    invA = np.linalg.inv(A)
    
    # 5. Оформление результатов
    print('L (Cholesky):', L, sep='\n')
    print('y:', y, sep='\n')
    print('x:', x, sep='\n')
    print('A^{-1}:', invA, sep='\n')
    
    # Output:
    # L (Cholesky):
    # [[ 2.  0.  0.]
    #  [ 6.  1.  0.]
    #  [-8.  5.  3.]]
    # y:
    # [ 0.5 -1.   4. ]
    # x:
    # [28.58333333 -7.66666667  1.33333333]
    # A^{-1}:
    # [[ 49.36111111 -13.55555556   2.11111111]
    #  [-13.55555556   3.77777778  -0.55555556]
    #  [  2.11111111  -0.55555556   0.11111111]]


L (Cholesky):
[[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]
y:
[ 0.5 -1.   4. ]
x:
[28.58333333 -7.66666667  1.33333333]
A^{-1}:
[[ 49.36111111 -13.55555556   2.11111111]
 [-13.55555556   3.77777778  -0.55555556]
 [  2.11111111  -0.55555556   0.11111111]]


In [29]:
# Матрица коэффициентов
A = np.array([
    [1,     1,    1],
    [8.0,  7.8,  7.7],
    [560, 580, 600]
])

# Вектор свободных членов
b = np.array([1, 7.85, 580])

# Решаем систему уравнений Ax = b
x = scipy.linalg.solve(A, b)

# Вывод решения
x_A, x_B, x_C = x
print(f"Решение системы: x_A = {x_A:.4f}, x_B = {x_B:.4f}, x_C = {x_C:.4f}")

# Output:
# Решение системы: x_A = 0.5000, x_B = 0.0000, x_C = 0.5000


Решение системы: x_A = 0.5000, x_B = 0.0000, x_C = 0.5000


# Практика

## Векторы

In [44]:
import sys


def main():
    
    k = int(input())
    k_num = list(map(float, input().split()))
    
    v_crds = []
    for i in range(k):
        v = list(map(float, input().split()))
        v_crds.append(v)

    n = len(v_crds[0])
    result = [0.0] * n
    
    for i in range(k):
        lam = k_num[i]
        vec = v_crds[i]
        for j in range(n):
            result[j] += lam * vec[j]
    
    print(" ".join(f"{x:.2f}" for x in result))

if __name__ == '__main__':
    main()


-3.00 -3.00 -3.00


In [4]:
import sys


def main():
    n = int(input())
    u = list(map(int, input().split()))
    v = list(map(int, input().split()))
    if n > 100:
        raise ValueError('n is not right!')
    for i in range(n):
        if u[i] > 1000 or v[i] > 1000:
            raise ValueError('u/v is not right!')
    
    result = 0
    for i in range(n):
        result += u[i] * v[i]

    if not result:
        print('ORTHOGONAL')
    else:
        print('NON-ORTHOGONAL')

if __name__ == '__main__':
    main()


NON-ORTHOGONAL


In [9]:
import sys


def main():
    
    # len = 2
    u1 = list(map(int, input().split()))
    u2 = list(map(int, input().split()))
    u3 = list(map(int, input().split()))
    for i in range(len(u1)):
        if u1[i] > 100 or u2[i] > 100 or u3[i] > 100:
            raise ValueError('vector is big')

    coeff = [i for i in range(-10, 11) if i != 0]

    for c1 in coeff:
        for c2 in coeff:
            if (c1 * u1[0] + c2 * u2[0] == u3[0] and
                c1 * u1[1] + c2 * u2[1] == u3[1]):
                print(c1, c2)
                return
            
    print('NO_SOLUTION')


if __name__ == '__main__':
    main()


NO_SOLUTION
